## Acknowledgements

To start, IвЂ™d like to thank **Tong Hui Kang** and **konbu17**.

The CoT data and part of the hyperparameter settings used in this notebook are adapted from Tong Hui KangвЂ™s open-source GitHub repository, while the training code is modified based on konbu17вЂ™s public notebook.  
- [Tong Hui KangвЂ™s open-source GitHub repository](https://github.com/tonghuikang/nemotron)
- [konbu17вЂ™s public notebook](https://www.kaggle.com/code/konbu17/nemotron-sft-lora-with-cot)

## Overview

In this notebook, I will try to reproduce the method released by Tong Hui Kang and train a model that can reach around **0.85** on the public leaderboard.

In this version, training on **"lm_head"** in target_modules has been **removed**. At the same time, the microbatch size has been reset to **1**. In my experiments, increasing the microbatch size to 2 reduced training time by about **3 hours**, but it also caused the LB score to drop by around **0.1**, and I have not yet been able to address this issue through hyperparameter tuning.

## Notes on Training

Setting per_device_train_batch_size to 2 and gradient_accumulation_steps to 16 can reduce the training time to around **4 hours** while maintaining similar performance **(0.84вЂ“0.85)**. However, given the nature of the competition, a score drop of about 0.1 is still difficult to accept.

It may be possible to restore training performance comparable to that of a microbatch size of 1 through **further hyperparameter tuning**.

I hope this notebook can also help others participate more effectively in this competition.

## Mode Selection

In [1]:
# ============================================================
# MODE SELECTION вЂ” set exactly one to 1
# ============================================================

import os, sys
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="strict")

# Mode A: Train LoRA from scratch on Kaggle GPU
TRAIN_ON_KAGGLE = 1

# Mode B: Use pre-trained LoRA weights from dataset and just package them
USE_PRETRAINED = 0

assert (TRAIN_ON_KAGGLE + USE_PRETRAINED) == 1, \
    "Set exactly one of TRAIN_ON_KAGGLE / USE_PRETRAINED to 1."

PRETRAINED_ADAPTER_DATASET_PATH = "/kaggle/input/datasets/konbu17/nemotron-sft-lora-cot-selection"
BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

print({
    "TRAIN_ON_KAGGLE": TRAIN_ON_KAGGLE,
    "USE_PRETRAINED": USE_PRETRAINED,
    "PRETRAINED_ADAPTER_DATASET_PATH": PRETRAINED_ADAPTER_DATASET_PATH,
})


{'TRAIN_ON_KAGGLE': 1, 'USE_PRETRAINED': 0, 'PRETRAINED_ADAPTER_DATASET_PATH': '/kaggle/input/datasets/konbu17/nemotron-sft-lora-cot-selection'}


## Setup & Model Loading

In [2]:
import os, glob, sys, subprocess, site

candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)

if not candidates:
    raise FileNotFoundError("No Triton wheel found under /kaggle/input")
wheel = candidates[0]

target = "/kaggle/working/pydeps"
os.makedirs(target, exist_ok=True)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-deps",
        "--target", target,
        "--upgrade",
        "--ignore-installed",
        wheel,
    ],
    check=True,
)

if target not in sys.path:
    sys.path.insert(0, target)

site.addsitedir(target)

print("Custom target added:", target)

import importlib.util
print("triton specпјљ", importlib.util.find_spec("triton"))


Found Triton wheels: ['/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl', '/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/triton-3.5.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl', '/kaggle/input/datasets/mayukh18/nemotron-packages/packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl']
Processing /kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
Custom target added: /kaggle/working/pydeps
triton specпјљ ModuleSpec(name='triton', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7e4d4caf0170>, origin='/kaggle/working/pydeps/triton/__init__.py', submodule_search_locations=['/kaggle/working/pydeps/triton'])


In [3]:
if TRAIN_ON_KAGGLE:
    import sys, os, shutil, stat

    # Add utility script to Python path (provides helper binaries)
    sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')

    # Copy ptxas-blackwell to /tmp with execute permissions
    ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
    ptxas_dst = '/tmp/ptxas-blackwell'
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        src_bin = os.path.dirname(ptxas_src)
        dst_bin = '/tmp/triton_nvidia_bin'
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst

        import triton.backends.nvidia as nv_backend
        nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst

    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: '12.0'

    print('Training environment fixes applied.')
else:
    print("USE_PRETRAINED=1: skipping Triton / ptxas environment fixes.")


Training environment fixes applied.


In [4]:
# trl installation is handled by the Unsloth offline setup cell below.
if TRAIN_ON_KAGGLE:
    print("Skip standalone trl install/import here; the Unsloth setup cell will install compatible packages.")

Skip standalone trl install/import here; the Unsloth setup cell will install compatible packages.


In [5]:
if TRAIN_ON_KAGGLE:
    import glob
    import os
    import subprocess
    import sys

    def recursive_wheels(pattern: str):
        return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

    packages_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
    all_mamba = recursive_wheels("mamba_ssm-*.whl")
    all_causal = recursive_wheels("causal*conv1d*.whl")

    print("Found mamba wheels:", all_mamba)
    print("Found causal-conv1d wheels:", all_causal)

    import torch
    print("Python:", sys.version)
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("Torch CUDA:", torch.version.cuda)

    if not torch.cuda.is_available():
        raise RuntimeError("TRAIN_ON_KAGGLE=1 requires a GPU runtime because Nemotron depends on CUDA wheels.")

    if not os.path.isdir(packages_dir):
        raise FileNotFoundError(f"Offline wheel directory not found: {packages_dir}")

    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "--no-index", "--find-links", packages_dir,
            "unsloth", "trl", "peft", "transformers", "datasets", "accelerate", "bitsandbytes",
        ],
        check=True,
    )

    def pick_last(wheels):
        return wheels[-1] if wheels else None

    causal_wheel = pick_last(all_causal)
    mamba_wheel = pick_last(all_mamba)
    print("Selected causal wheel:", causal_wheel)
    print("Selected mamba wheel:", mamba_wheel)

    if causal_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", causal_wheel], check=True)
    if mamba_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", mamba_wheel], check=True)
    else:
        raise FileNotFoundError("Could not find a compatible mamba_ssm wheel under /kaggle/input.")

    print("Offline package installation finished. Restart the kernel if Kaggle keeps stale imports from earlier runs.")
else:
    print("USE_PRETRAINED=1: skipping datasets / trl / mamba_ssm / unsloth installation.")


Found mamba wheels: ['/kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl']
Found causal-conv1d wheels: ['/kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl']
Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Torch: 2.10.0+cu128
CUDA available: True
Torch CUDA: 12.8


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.9.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.9.0 which is incompatible.


Selected causal wheel: /kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
Selected mamba wheel: /kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
Processing /kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
Processing /kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
Offline package installation finished. Restart the kernel if Kaggle keeps stale imports from earlier runs.


In [6]:
if TRAIN_ON_KAGGLE:
    import torch
    import kagglehub
    from unsloth import FastLanguageModel

    MAX_SEQ_LEN = 8192
    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Model path: {MODEL_PATH}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=False,
        load_in_8bit=False,
        full_finetuning=False,
        trust_remote_code=True,
        unsloth_force_compile=False,
        attn_implementation="eager",
        dtype=torch.bfloat16,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print("Model loaded with Unsloth.")
else:
    print("USE_PRETRAINED=1: skipping base model and tokenizer loading.")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-05-01 21:52:06.878717: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777672327.064003      65 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777672327.117953      65 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777672327.590716      65 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777672327.590734      65 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777672327.590735      65 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
Model path: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.3.17: Fast Nemotron_H patching. Transformers: 4.57.6.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading checkpoint shards:   0%|          | 0/13 [00:00<?, ?it/s]

/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1 does not have a padding token! Will use pad_token = <SPECIAL_999>.
Model loaded with Unsloth.


In [7]:
if TRAIN_ON_KAGGLE:
    from unsloth import FastLanguageModel

    LORA_RANK = 32
    LORA_ALPHA = 32
    LORA_DROPOUT = 0.0
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "in_proj", "out_proj", "up_proj", "down_proj",
    ]

    print("Creating trainable LoRA wrapper via FastLanguageModel.get_peft_model ...")
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=target_modules,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )
    model.print_trainable_parameters()
else:
    print("USE_PRETRAINED=1: skipping trainable LoRA construction.")


Creating trainable LoRA wrapper via FastLanguageModel.get_peft_model ...
Unsloth: Detected MoE model with num_experts = 128 and target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'in_proj', 'out_proj', 'up_proj', 'down_proj']. Enabling LoRA on MoE parameters: ['mlp.experts.gate_up_proj', 'mlp.experts.down_proj']
Unsloth: PEFT set target_parameters but found no matching parameters.
This is expected for MoE models - Unsloth handles MoE expert LoRA targeting separately.
Unsloth: Making `model.base_model.model.backbone` require gradients
trainable params: 883,873,792 || all params: 32,461,811,136 || trainable%: 2.7228


## Mode A: Train on Kaggle

In [8]:
if TRAIN_ON_KAGGLE:
    import pandas as pd
    import random
    import gc, time
    from datasets import Dataset as HFDataset
    from trl import SFTTrainer, SFTConfig

    SEED = 42
    PROMPT_SUFFIX = '\nPlease solve carefully. Put exactly one final answer inside `\\boxed{}` at the very end. Do not use `\\boxed{}` for intermediate values. After solving, verify your answer with an independent check.'
    MISMATCH_POLICY = "short"  # one of: "drop", "short"

    DATASET_PATH = "/kaggle/input/datasets/dgxchen/nemotron-cot-tong/problem_ids_matched.csv"
    df = pd.read_csv(DATASET_PATH)
    print(f"Full dataset: {len(df)} rows")

    train_df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    print(f"Full dataset: {len(df)} rows")

    import re
    import math
    from collections import Counter, defaultdict
    from torch.utils.data import DataLoader, Sampler

    def normalize_answer(value):
        text = "" if value is None else str(value).strip()
        text = text.replace("$", "")
        text = re.sub(r"\s+", " ", text)
        return text.strip()

    def extract_final_answer(text: str) -> str:
        """Extract the final answer using the same priority order as the Kaggle metric."""
        if text is None:
            return "NOT_FOUND"
        text = str(text)

        matches = re.findall(r'\\boxed\{([^}]*)\}', text)
        non_empty = [m.strip() for m in matches if m.strip()]
        if non_empty:
            return non_empty[-1]
        if matches:
            return matches[-1].strip()

        patterns = [
            r'The final answer is:\s*([^\n]+)',
            r'Final answer is:\s*([^\n]+)',
            r'Final answer\s*[:：]\s*([^\n]+)',
            r'final answer\s*[:：]\s*([^\n]+)',
        ]
        for pattern in patterns:
            found = re.findall(pattern, text, re.IGNORECASE)
            if found:
                return found[-1].strip()

        found = re.findall(r'-?\d+(?:\.\d+)?', text)
        if found:
            return found[-1]

        lines = [line.strip() for line in text.splitlines() if line.strip()]
        return lines[-1] if lines else "NOT_FOUND"

    def answers_match(expected, predicted) -> bool:
        expected = normalize_answer(expected)
        predicted = normalize_answer(predicted)
        try:
            expected_num = float(expected)
            predicted_num = float(predicted)
            return math.isclose(expected_num, predicted_num, rel_tol=1e-2, abs_tol=1e-2)
        except Exception:
            pass
        return expected.lower() == predicted.lower()

    def strip_boxed_commands(text: str) -> str:
        """Remove every intermediate \\boxed{...} command while preserving its content."""
        if text is None:
            return ""
        text = str(text)
        out = []
        i = 0
        marker = r"\boxed{"
        while i < len(text):
            j = text.find(marker, i)
            if j == -1:
                out.append(text[i:])
                break
            out.append(text[i:j])
            k = j + len(marker)
            depth = 1
            inner = []
            while k < len(text) and depth > 0:
                ch = text[k]
                if ch == "{":
                    depth += 1
                    inner.append(ch)
                elif ch == "}":
                    depth -= 1
                    if depth > 0:
                        inner.append(ch)
                else:
                    inner.append(ch)
                k += 1
            if depth == 0:
                out.append("".join(inner).strip())
                i = k
            else:
                i = j + len(r"\boxed")
        cleaned = "".join(out)
        cleaned = re.sub(r"\\boxed\s*", "", cleaned)
        cleaned = re.sub(r"\\fbox\s*", "", cleaned)
        cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)
        return cleaned.strip()

    def build_type_aware_verifier(problem_type: str, answer: str) -> str:
        problem_type = str(problem_type)
        answer = normalize_answer(answer)
        if "unit_conversion" in problem_type:
            checks = [
                "re-check the conversion direction and scale factor",
                "convert the result back to the original unit to confirm consistency",
            ]
        elif "bit_manipulation" in problem_type:
            checks = [
                "recompute the bit operation from the binary representation",
                "confirm the decimal value after the final bit state is obtained",
            ]
        elif "cipher" in problem_type:
            checks = [
                "verify that the same mapping is used for every symbol",
                "decode the relevant text once more and compare it with the derived answer",
            ]
        elif "numeral" in problem_type:
            checks = [
                "convert through place values/base notation carefully",
                "convert the candidate answer back to confirm the representation",
            ]
        elif "equation_numeric" in problem_type:
            checks = [
                "substitute the candidate value back into the equation",
                "check that the numerical residual is within tolerance",
            ]
        elif "cryptarithm" in problem_type:
            checks = [
                "check that every letter has one consistent digit",
                "substitute the assignment back into the arithmetic constraint",
            ]
        elif "gravity" in problem_type:
            checks = [
                "verify the formula, signs, and units used in the computation",
                "estimate the magnitude to ensure the result is physically plausible",
            ]
        else:
            checks = [
                "solve the problem a second way or reverse the computation",
                "confirm that the final candidate satisfies the original prompt",
            ]
        return (
            "Independent check:\n"
            f"- I {checks[0]}.\n"
            f"- I {checks[1]}.\n"
            f"- The checked answer is {answer}."
        )

    records = []
    record_types = []
    cot_quality = Counter()
    type_quality = defaultdict(Counter)
    for _, row in train_df.iterrows():
        prompt = str(row["prompt"])
        answer = normalize_answer(row["answer"])
        problem_type = str(row["type"])
        cot = str(row["generated_cot"])
        if not cot or cot == "nan" or len(cot.strip()) < 5:
            cot_quality["missing_cot"] += 1
            type_quality[problem_type]["missing_cot"] += 1
            continue

        cot_answer = extract_final_answer(cot)
        is_good_cot = answers_match(answer, cot_answer)
        quality_label = "good_cot" if is_good_cot else "suspicious_cot"
        cot_quality[quality_label] += 1
        type_quality[problem_type][quality_label] += 1

        verifier_block = build_type_aware_verifier(problem_type, answer)
        user_content = prompt + PROMPT_SUFFIX

        if is_good_cot:
            cot_cleaned = strip_boxed_commands(cot).rstrip()
            assistant_content = cot_cleaned + f"\n\n{verifier_block}\n</think>\n\\boxed{{{answer}}}"
        else:
            if MISMATCH_POLICY == "drop":
                cot_quality["dropped_suspicious_cot"] += 1
                type_quality[problem_type]["dropped_suspicious_cot"] += 1
                continue
            if MISMATCH_POLICY != "short":
                raise ValueError(f"Unsupported MISMATCH_POLICY={MISMATCH_POLICY!r}")
            assistant_content = (
                "The draft chain of thought was inconsistent with the verified target, so I use a short checked solution.\n\n"
                f"{verifier_block}\n</think>\n\\boxed{{{answer}}}"
            )

        reasoning_part = assistant_content.rsplit("\\boxed{", 1)[0]
        if r"\boxed" in reasoning_part or r"\fbox" in reasoning_part:
            raise ValueError("Intermediate boxed answer survived sanitization")

        records.append({"messages": [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": assistant_content},
        ]})
        record_types.append(problem_type)

    dataset = HFDataset.from_list(records)
    print(f"SFT records: {len(records)}")
    print("CoT quality:", dict(cot_quality))
    print("CoT quality by type:", {k: dict(v) for k, v in sorted(type_quality.items())})

    def formatting_prompts_func(example):
        messages = example["messages"]
        if messages and isinstance(messages[0], dict):
            conversations = [messages]
        else:
            conversations = messages

        texts = []
        for conversation in conversations:
            try:
                text = tokenizer.apply_chat_template(
                    conversation,
                    tokenize=False,
                    add_generation_prompt=False,
                    enable_thinking=True,
                )
            except TypeError:
                text = tokenizer.apply_chat_template(
                    conversation,
                    tokenize=False,
                    add_generation_prompt=False,
                )
            texts.append(text)
        return texts


    training_args = SFTConfig(
        output_dir="/kaggle/working/sft_output",
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=32,
        learning_rate=2e-4,
        lr_scheduler_type="linear",
        warmup_steps=0,
        max_length=8192,
        adam_beta1=0.9,
        adam_beta2=0.95,
        adam_epsilon=1e-8,
        weight_decay=0.0,
        max_grad_norm=1e9,
        logging_steps=10,
        save_strategy="no",
        bf16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        dataloader_num_workers=2,
        remove_unused_columns=False,
        seed=SEED,
        report_to="none",
        packing=False,
    )

    def build_stratified_index_order(labels, batch_size, seed):
        """Approximate nemotron-master's stratified batching over effective batches."""
        by_label = defaultdict(list)
        for idx, label in enumerate(labels):
            by_label[label].append(idx)

        rng = random.Random(seed)
        for idx_list in by_label.values():
            rng.shuffle(idx_list)

        n_batches = max(1, math.ceil(len(labels) / batch_size))
        batches = [[] for _ in range(n_batches)]
        batch_order = list(range(n_batches))
        rng.shuffle(batch_order)

        assigned = 0
        for label in sorted(by_label.keys()):
            for idx in by_label[label]:
                batches[batch_order[assigned % n_batches]].append(idx)
                assigned += 1

        order = [idx for batch in batches for idx in batch]
        if len(order) != len(labels):
            raise ValueError("Stratified order size mismatch")
        return order

    class PrecomputedOrderSampler(Sampler):
        def __init__(self, order):
            self.order = list(order)

        def __iter__(self):
            return iter(self.order)

        def __len__(self):
            return len(self.order)

    class StratifiedSFTTrainer(SFTTrainer):
        def __init__(self, *args, stratified_order=None, **kwargs):
            super().__init__(*args, **kwargs)
            self.stratified_order = stratified_order

        def get_train_dataloader(self):
            if self.train_dataset is None:
                raise ValueError("Trainer requires a train_dataset.")
            if self.stratified_order is None:
                return super().get_train_dataloader()
            if len(self.stratified_order) != len(self.train_dataset):
                raise ValueError("Stratified order length does not match train dataset")

            dataloader_kwargs = {
                "batch_size": self.args.per_device_train_batch_size,
                "sampler": PrecomputedOrderSampler(self.stratified_order),
                "collate_fn": self.data_collator,
                "num_workers": self.args.dataloader_num_workers,
                "pin_memory": self.args.dataloader_pin_memory,
                "persistent_workers": self.args.dataloader_persistent_workers,
                "drop_last": self.args.dataloader_drop_last,
            }
            if self.args.dataloader_num_workers > 0:
                dataloader_kwargs["prefetch_factor"] = self.args.dataloader_prefetch_factor

            return DataLoader(self.train_dataset, **dataloader_kwargs)

    effective_batch_size = max(
        1,
        training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps,
    )
    stratified_order = build_stratified_index_order(record_types, effective_batch_size, SEED)
    print(f"Approx stratified effective batch size: {effective_batch_size}")
    print("Stratified batching by type:", dict(sorted(pd.Series(record_types).value_counts().to_dict().items())))

    trainer = StratifiedSFTTrainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        processing_class=tokenizer,
        formatting_func=formatting_prompts_func,
        stratified_order=stratified_order,
    )

    print("Starting SFT training...")
    t0 = time.time()
    trainer.train()
    elapsed = time.time() - t0
    print(f"Training done in {elapsed/60:.1f} min")


    ADAPTER_DIR = "/kaggle/working/sft_adapter"
    model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    print(f"Adapter saved to {ADAPTER_DIR}")



Full dataset: 7830 rows
Full dataset: 7830 rows
SFT records: 7830
CoT quality: {'good_cot': 7830}
CoT quality by type: {'bit_manipulation': {'good_cot': 1754}, 'cipher': {'good_cot': 1656}, 'cryptarithm_deduce': {'good_cot': 627}, 'cryptarithm_guess': {'good_cot': 154}, 'equation_numeric_deduce': {'good_cot': 658}, 'equation_numeric_guess': {'good_cot': 126}, 'gravity': {'good_cot': 1055}, 'numeral': {'good_cot': 730}, 'unit_conversion': {'good_cot': 1070}}
Approx stratified effective batch size: 32
Stratified batching by type: {'bit_manipulation': 1754, 'cipher': 1656, 'cryptarithm_deduce': 627, 'cryptarithm_guess': 154, 'equation_numeric_deduce': 658, 'equation_numeric_guess': 126, 'gravity': 1055, 'numeral': 730, 'unit_conversion': 1070}


Unsloth: Tokenizing ["text"] (num_proc=52):   0%|          | 0/7830 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Starting SFT training...
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,20.262200
20,13.854800
30,6.678200
40,3.181500
50,2.171700
60,1.939200
70,1.750300
80,1.674800
90,1.657100
100,1.507400


Training done in 408.0 min
Adapter saved to /kaggle/working/sft_adapter


## Mode B: Load Pre-trained LoRAпј€Temporarily unavailableпј‰

In [9]:
if USE_PRETRAINED:
    import os

    SRC_ADAPTER_DIR = PRETRAINED_ADAPTER_DATASET_PATH
    required_files = ["adapter_config.json", "adapter_model.safetensors"]

    print("Using pre-trained adapter from:", SRC_ADAPTER_DIR)
    for fname in required_files:
        fpath = os.path.join(SRC_ADAPTER_DIR, fname)
        if not os.path.exists(fpath):
            raise FileNotFoundError(f"Missing required adapter file: {fpath}")
        print(f"  {fname}: {os.path.getsize(fpath)/1024/1024:.1f} MB")
else:
    print("TRAIN_ON_KAGGLE=1: pretrained adapter path check skipped.")


TRAIN_ON_KAGGLE=1: pretrained adapter path check skipped.


## Create submission.zip

In [10]:
import json, os, shutil, zipfile

OUTPUT_DIR = "/kaggle/working"
SUBMISSION_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "submission_adapter")
os.makedirs(SUBMISSION_ADAPTER_DIR, exist_ok=True)

required_files = ["adapter_config.json", "adapter_model.safetensors"]

if TRAIN_ON_KAGGLE:
    src_adapter_dir = "/kaggle/working/sft_adapter"
    print("Packaging freshly trained adapter from:", src_adapter_dir)
else:
    src_adapter_dir = PRETRAINED_ADAPTER_DATASET_PATH
    print("Packaging pre-trained adapter directly from:", src_adapter_dir)

for fname in required_files:
    src = os.path.join(src_adapter_dir, fname)
    dst = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
    if not os.path.exists(src):
        raise FileNotFoundError(f"Missing required adapter file: {src}")
    shutil.copy2(src, dst)
    print(f"Copied {fname} ({os.path.getsize(dst)/1024/1024:.1f} MB)")

config_path = os.path.join(SUBMISSION_ADAPTER_DIR, "adapter_config.json")
with open(config_path, "r") as f:
    cfg = json.load(f)

cfg["base_model_name_or_path"] = BASE_MODEL_NAME
cfg["inference_mode"] = True
cfg["lora_dropout"] = 0.0

with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

zip_path = os.path.join(OUTPUT_DIR, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        fpath = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
        zf.write(fpath, fname)
        print(f"  Added {fname}")

zip_sz = os.path.getsize(zip_path) / 1024 / 1024
print(f"\nsubmission.zip: {zip_sz:.1f} MB")
print("Done! Ready to submit.")


Packaging freshly trained adapter from: /kaggle/working/sft_adapter
Copied adapter_config.json (0.0 MB)
Copied adapter_model.safetensors (3373.4 MB)
  Added adapter_config.json
  Added adapter_model.safetensors

submission.zip: 3093.2 MB
Done! Ready to submit.
